# ☀️ Desafío Evaluado (Ev. 3 y Ev. 4): Segmentación y Clasificación de Datos Fotovoltaicos
**Planta Solar Lechuzas (`10 - lechuzasdataset.csv` — 105,216 registros)**

---
## 1. Mapa Conceptual Teórico: Clusterización (K-Means vs. Dendrograma) vs. Clasificación Supervisada

| Dimensión | K-Means (Particional - Seleccionado) | Clustering Jerárquico / Dendrograma | Clasificación Supervisada (Random Forest) |
| :--- | :--- | :--- | :--- |
| **Paradigma** | Aprendizaje No Supervisado | Aprendizaje No Supervisado | Aprendizaje Supervisado |
| **Principio Matemático** | Minimiza la inercia intra-cluster $WCSS = \sum_{k=1}^K \sum_{x_i \in C_k} \|x_i - \mu_k\|^2$ | Construye árbol de fusiones sobre matriz de distancias $N \times N$ (enlace *Ward*, *Complete*, *Average*) | Ensamble de 100 árboles de decisión entrenados sobre las etiquetas de cluster descubiertas |
| **Escalabilidad ($N=105,216$)** | **Alta**: Complejidad lineal $O(N \cdot K \cdot I \cdot d)$ (< 1 segundo) | **Baja**: Matriz $105,216^2$ requiere ~88 GB de RAM ($O(N^2)$) | **Alta**: Inferencia en tiempo real (< 5 ms por consulta en producción) |
| **Métricas Clave** | Inercia (Codo), *Silhouette Score*, *Davies-Bouldin*, *Calinski-Harabasz* | Distancia cophenética, corte de dendrograma, *Silhouette* | *Accuracy*, *Precision*, *Recall*, *F1-Score*, Matriz de Confusión |

## 2. Carga de Datos y Análisis Exploratorio (EDA)

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

df = pd.read_csv('10 - lechuzasdataset.csv')
df.columns = df.columns.str.strip()
print('Dimensiones:', df.shape)
print('Valores nulos:\n', df.isnull().sum())
df.describe()

## 3. Preprocesamiento Físico y Escalamiento de Variables
1. **Eliminación de `id`**: Es un índice correlativo (`0` a `105,215`) que distorsionaría la distancia euclidiana.
2. **Corrección de valores negativos en `Radiacion`**: Durante la noche existen `45,142` lecturas con valores entre `-1.0` y `-10.0 W/m²` producto del offset térmico nocturno del piranómetro. Físicamente la irradiancia solar no puede ser negativa, por lo que se truncan a `0.0 W/m²`.
3. **Escalamiento con `StandardScaler`**: Indispensable para K-Means, igualando todas las variables a media `0` y desviación estándar `1`.

In [ ]:
df_clean = df.drop(columns=['id']).copy()
df_clean['Radiacion'] = df_clean['Radiacion'].clip(lower=0.0)

FEATURE_COLS = ['Potencia', 'Radiacion', 'Temperatura', 'Temperatura panel']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_clean[FEATURE_COLS].values)
df_clean.describe()

## 4. Justificación de Hiperparámetros ($k = 2..6$): Método del Codo y Métricas Internas

In [ ]:
resultados_k = []
for k in [2, 3, 4, 5, 6]:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    resultados_k.append({
        'k': k,
        'Inercia_WCSS': km.inertia_,
        'Silhouette': silhouette_score(X_scaled, labels, sample_size=10000, random_state=42),
        'Davies_Bouldin': davies_bouldin_score(X_scaled, labels),
        'Calinski_Harabasz': calinski_harabasz_score(X_scaled, labels)
    })

pd.DataFrame(resultados_k)

## 5. Implementación de K-Means ($k = 4$) e Interpretación Física de Regímenes Solares

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
raw_clusters = kmeans.fit_predict(X_scaled)

# Ordenar clusters de menor a mayor Radiación solar promedio
orden = np.argsort([df_clean.loc[raw_clusters == c, 'Radiacion'].mean() for c in range(4)])
mapa = {old_c: int(new_c) for new_c, old_c in enumerate(orden)}
df_clean['cluster'] = np.vectorize(mapa.get)(raw_clusters)

NOMBRES = {
    0: 'Operación Nocturna / Standby',
    1: 'Baja Radiación / Transición',
    2: 'Generación Media / Templada',
    3: 'Generación Pico / Alta Temp. Panel'
}
df_clean['regimen_operativo'] = df_clean['cluster'].map(NOMBRES)
df_clean.groupby(['cluster', 'regimen_operativo'])[FEATURE_COLS].agg(['count', 'mean'])

## 6. Automatización para Producción: Entrenamiento y Serialización del Clasificador Supervisado

In [ ]:
X = df_clean[FEATURE_COLS]
y = df_clean['cluster']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler_prod = StandardScaler()
X_train_sc = scaler_prod.fit_transform(X_train.values)
X_test_sc = scaler_prod.transform(X_test.values)

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_sc, y_train)

y_pred = rf.predict(X_test_sc)
print(f'Accuracy en Producción: {accuracy_score(y_test, y_pred):.4f}')
print(classification_report(y_test, y_pred))

os.makedirs('models', exist_ok=True)
joblib.dump(rf, 'models/clasificador_solar_rf.pkl')
joblib.dump(scaler_prod, 'models/scaler_solar.pkl')